# 3. CropHarvest: prepare references and collect annual imagery

Audit carrot, cabbage and potato labels before selecting candidate parcels. Source reliability and date checks remain unchanged.


## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Existing settings, selections, archives and jobs are preserved.

- `reuse`: use local references; satellite collection is enabled separately.
- `download_missing`: obtain missing references within the source limits.
- `offline`: use local references, with no package installation, sign-in or satellite calls. Dependencies must already be installed.

Use `download_missing` on a fresh clone. Enter an Earth Engine-enabled project ID when prompted. All sources target **up to 100 candidate parcels per available crop per round**; existing larger batches remain intact. Saved outputs may describe earlier runs until cells are rerun.


In [ ]:
# Review these settings before Run All.
reference_mode = "reuse"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls

from pathlib import Path
import os
import sys
repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
from crop_pipeline.windows.seasons import file_sha256
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection
download_workers = 2
intervals_per_request = 6

from crop_pipeline.sources import cropharvest as cropharvest_source, cropharvest_review
from crop_pipeline.imagery.quality import save_json
from IPython.display import display
source_name = "cropharvest"
inward_buffer_metres = 10  # Polygon interiors, preserving the existing rule.
labels_path = repository_root / "data/external/cropharvest/labels.geojson"
country_boundaries_path = repository_root / "data/external/cropharvest/ne_50m_admin_0_countries.zip"
minimum_reference_year, maximum_reference_year = 2017, 2100
observation_year_overrides = {}  # Explicit source-UID decisions for unresolved dates.

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
print(f"Source: {source_name} | References: {reference_mode} | Round limit: {candidate_parcels_per_crop} per crop")
print(f"Inward buffer: {inward_buffer_metres} m | Satellite collection: {run_satellite_collection}")
print("Archive:", annual_archive_root)
print("Backend: Earth Engine batch assets → Parquet transport → annual TIFFs")
notebook_progress = pipeline_progress.start_notebook(
    '03_cropharvest_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)


## 2. Obtain references

Reuse and validate `labels.geojson`, or obtain it in `download_missing` mode from the [pinned release](https://zenodo.org/records/10251170). New downloads are checksum-checked and saved atomically. No imagery/feature archive is downloaded. Country borders are optional; dataset names provide the existing fallback.


In [ ]:
pipeline_progress.begin_stage("Obtain references")
labels_download = cropharvest_source.ensure_labels(
    labels_path, download_if_missing=reference_options["download_if_missing"],
)
labels_revision = "labels.geojson sha256:" + file_sha256(labels_path)
print("Labels:", labels_download["status"], "| features:", labels_download["features"])
reference_note = "Pinned CropHarvest labels; source grades and dates are screening evidence"
pipeline_progress.end_stage()


## 3. Validate references

Normalize crop names, resolve observation years, and apply the existing source-reliability, geometry, duplicate and season checks. Save the full label and year audits; show one crop-level summary. Exclude upstream test labels before preparing footprints.

Dates establish reference years, not planting dates. The helper preserves existing eligibility rules. Polygon interiors use the existing **10 m inward buffer**.


In [ ]:
pipeline_progress.begin_stage("Validate references")
# Version the audit independently of notebook layout and collection workload.
run_configuration = {
    "labels_sha256": file_sha256(labels_path),
    "country_boundaries_sha256": file_sha256(country_boundaries_path) if country_boundaries_path.exists() else None,
    "year_range": [minimum_reference_year, maximum_reference_year],
    "observation_year_overrides": observation_year_overrides,
    "review_helper_sha256": file_sha256(Path(cropharvest_review.__file__)),
}
run_folder, run_revision = cropharvest_source.initialize_run(
    annual_archive_root / "cropharvest_label_review", run_name, run_configuration,
)
label_review = cropharvest_review.review_labels(
    labels_path, country_boundaries_path, run_folder,
    minimum_reference_year=minimum_reference_year, maximum_reference_year=maximum_reference_year,
    observation_year_overrides=observation_year_overrides,
)
reviewed_labels = label_review["reviewed_labels"]
display(label_review["summary"])
print("Year decisions:", label_review["observation_year_audit"])
print("Screened labels:", label_review["screened_labels_output"])
eligible_labels = reviewed_labels.loc[reviewed_labels["reliable"].eq(True) & ~reviewed_labels["is_test"].eq(True)].copy()
eligible_labels["year"] = eligible_labels["observation_year"].astype(int)
reference_features, footprint_exclusions = [], []
for record in eligible_labels.itertuples():
    try:
        reference_features.append(cropharvest_source.parcel_feature(
            record.uid, record.geometry, record.crop, record.year, record.country,
            record.dataset, labels_revision,
        ))
    except ValueError as error:
        footprint_exclusions.append({"uid": record.uid, "reason": str(error)})
save_json(run_folder / "labels/footprint_exclusions.json", footprint_exclusions)
print("Eligible footprints:", len(reference_features), "| exclusions:", len(footprint_exclusions))
print("Edge handling: 10 m inward buffer on parcel polygons.")
pipeline_progress.end_stage()


## 4. Prepare the collection round

Resume original saved plans and jobs. Fill the round limit only with previously unselected parcels for deficient crop/partition combinations; locked partitions stay unchanged. Prepare masks only for selected candidates and show the combined workload. Source-specific edge handling is preserved.


In [ ]:
pipeline_progress.begin_stage("Prepare collection round")
adaptive_batches = adaptive.prepare_round_batches(
    "cropharvest", reference_features, annual_archive_root, report_folder,
    provenance={"labels_revision": labels_revision, "label_review_revision": run_revision,
                "label_review_folder": str(run_folder), "footprint_exclusions": footprint_exclusions},
    inward_buffer_metres=inward_buffer_metres, parcels_per_crop=candidate_parcels_per_crop,
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Active/resumed batches:", len(adaptive_batches))
print("Selected parcels:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Tiles:", sum(plan["estimate"]["tiles"] for plan, context in adaptive_batches if plan is not None))
print("Years:", sorted({year for plan, context in adaptive_batches if plan is not None for year in plan["years"]}))
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()


## 5. Collect annual imagery

Submit or collect existing Earth Engine jobs. Keep the laptop on until Run All returns normally. Submitted jobs continue in the cloud; rerun later to retrieve results and create annual TIFFs.

Cloud masks, daily mosaics, bilinear band alignment and half-month medians are unchanged. Each completed tile retains 24 periods × seven measurements. One progress display follows the active step; detailed task/page messages are saved to its log. Completion describes saved data, not usable-window quality.


In [ ]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


## 6. Summary and next action

Check parcel counts, verified tiles, cloud job states and saved paths below. Follow the reported next action. Finish launched batches before [04 Four-month windows](04_four_month_windows.ipynb); it assesses usable samples and deficits. Continue to 05–06 only after targets are met.

Annual TIFFs are under `annual_tifs/<source>/<year>/<campaign>/round_<number>/`, beside the configured archive root. Masks and Parquet caches remain separate. See [the workflow guide](../docs/workflow.md).


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
source_workflow.print_summary(
    source_name, adaptive_batches, collection_result, reference_mode,
    inward_buffer_metres, reference_note=reference_note,
)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
